In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('yolov8n.pt')



vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\stock-footage-cars-parking-and-leaving-cctv-feed.webm")

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        results = model(frame, classes=[1, 2, 3, 5, 7])

        for result in results:
            boxes = result.boxes.xyxy
            confs = result.boxes.conf
            classIds = result.boxes.cls

            for box, conf, classId in zip(boxes, confs, classIds):
                x1, y1, x2, y2 = map(int, box)

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

        cv2.imshow("video", frame)
        cv2.waitKey(40)

vid.release()
cv2.destroyAllWindows


In [7]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

while True:
    ret, frame = vid.read()
    if not ret:
        break

    results = model(frame, classes=[1, 2, 3, 5, 7])  # cars, motorbikes, buses, trucks

    for result in results:
        boxes = result.boxes.xyxy

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            # Extract detected car region
            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

            # Mask out extreme dark/light pixels (shadows, glass, headlights)
            sat_mask = hsv_roi[:, :, 1] > 20
            val_mask = (hsv_roi[:, :, 2] > 40) & (hsv_roi[:, :, 2] < 250)
            valid_mask = sat_mask & val_mask

            if np.count_nonzero(valid_mask) == 0:
                continue

            # Average HSV of valid pixels
            avg_hue = np.mean(hsv_roi[:, :, 0][valid_mask])
            avg_sat = np.mean(hsv_roi[:, :, 1][valid_mask])
            avg_val = np.mean(hsv_roi[:, :, 2][valid_mask])

            # Determine color
            colour = "Undefined"
            if avg_val < 40:
                colour = "BLACK"
            elif avg_sat < 40 and avg_val > 200:
                colour = "WHITE"
            elif avg_sat < 40 and 40 <= avg_val <= 200:
                colour = "GRAY"
            elif avg_hue < 5 or avg_hue >= 170:
                colour = "RED"
            elif avg_hue < 22:
                colour = "ORANGE"
            elif avg_hue < 78:
                colour = "YELLOW"
            elif avg_hue < 131:
                colour = "BLUE"
            else:
                colour = "VIOLET"

            # Draw bounding box and label
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, colour, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("video", frame)
    if cv2.waitKey(40) & 0xFF == 27:  # ESC to exit
        break

vid.release()
cv2.destroyAllWindows()



0: 384x640 2 cars, 52.4ms
Speed: 1.1ms preprocess, 52.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 49.6ms
Speed: 1.0ms preprocess, 49.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 67.0ms
Speed: 1.3ms preprocess, 67.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 45.7ms
Speed: 1.0ms preprocess, 45.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 51.5ms
Speed: 0.9ms preprocess, 51.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 51.8ms
Speed: 1.3ms preprocess, 51.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.2ms
Speed: 1.0ms preprocess, 44.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 53.0ms
Speed: 1.0ms preprocess, 53.0ms inference, 1.3ms postprocess

KeyboardInterrupt: 

In [6]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

while True:
    ret, frame = vid.read()
    if not ret:
        break

    results = model(frame, classes=[1, 2, 3, 5, 7])  # cars, motorbikes, buses, trucks

    for result in results:
        boxes = result.boxes.xyxy

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            roi = frame[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

            # Looser masking: only remove extreme dark/bright pixels
            val = hsv_roi[:, :, 2]
            valid_mask = (val > 5) & (val < 250)

            if np.count_nonzero(valid_mask) == 0:
                continue

            # Average HSV of valid pixels
            avg_hue = np.mean(hsv_roi[:, :, 0][valid_mask])
            avg_sat = np.mean(hsv_roi[:, :, 1][valid_mask])
            avg_val = np.mean(val[valid_mask])

            # Determine color
            colour = "Undefined"
            if avg_val < 40:
                colour = "BLACK"
            elif avg_sat < 50 and avg_val > 180:
                colour = "WHITE"
            elif avg_sat < 50 and 40 <= avg_val <= 180:
                colour = "GRAY"
            elif avg_hue < 5 or avg_hue >= 170:
                colour = "RED"
            elif avg_hue < 22:
                colour = "ORANGE"
            elif avg_hue < 78:
                colour = "YELLOW"
            elif avg_hue < 131:
                colour = "BLUE"
            else:
                colour = "VIOLET"

            # Draw bounding box and label
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, colour, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("video", frame)
    if cv2.waitKey(40) & 0xFF == 27:  # ESC to exit
        break

vid.release()
cv2.destroyAllWindows()



0: 384x640 2 cars, 54.4ms
Speed: 3.0ms preprocess, 54.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 50.4ms
Speed: 1.0ms preprocess, 50.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 50.9ms
Speed: 1.3ms preprocess, 50.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 49.4ms
Speed: 1.0ms preprocess, 49.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 52.8ms
Speed: 1.0ms preprocess, 52.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.1ms
Speed: 1.0ms preprocess, 44.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 52.2ms
Speed: 1.2ms preprocess, 52.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.7ms
Speed: 0.9ms preprocess, 44.7ms inference, 0.9ms postprocess

In [9]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\istockphoto-1142262134-640_adpp_is.mp4")

# Define color ranges in HSV (you can adjust these based on your needs)
COLOR_RANGES = {
    'red': [(0, 50, 50), (10, 255, 255)],
    'red2': [(170, 50, 50), (180, 255, 255)],  # Red wraps around 180°
    'blue': [(100, 50, 50), (140, 255, 255)],
    'green': [(40, 50, 50), (80, 255, 255)],
    'yellow': [(20, 50, 50), (40, 255, 255)],
    'white': [(0, 0, 200), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 30)],
}

def get_dominant_color(hsv_roi):
    """Get the dominant color from HSV ROI"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Calculate average HSV values
    avg_h = np.mean(hsv_roi[:, :, 0])
    avg_s = np.mean(hsv_roi[:, :, 1])
    avg_v = np.mean(hsv_roi[:, :, 2])
    
    # Determine color based on HSV ranges
    if avg_v < 50:
        return "black"
    elif avg_s < 30 and avg_v > 200:
        return "white"
    elif (avg_h >= 0 and avg_h <= 10) or (avg_h >= 170 and avg_h <= 180):
        return "red"
    elif avg_h >= 20 and avg_h <= 40:
        return "yellow"
    elif avg_h >= 40 and avg_h <= 80:
        return "green"
    elif avg_h >= 100 and avg_h <= 140:
        return "blue"
    else:
        return "unknown"

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])

        for result in results:
            boxes = result.boxes.xyxy
            confs = result.boxes.conf
            classIds = result.boxes.cls

            for box, conf, classId in zip(boxes, confs, classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Extract ROI from HSV frame
                roi_hsv = hsv_frame[y1:y2, x1:x2]
                
                # Get dominant color
                color_name = get_dominant_color(roi_hsv)
                
                # Draw rectangle with color-specific bounding box
                if color_name == "red":
                    color = (0, 0, 255)  # Red
                elif color_name == "blue":
                    color = (255, 0, 0)  # Blue
                elif color_name == "green":
                    color = (0, 255, 0)  # Green
                elif color_name == "yellow":
                    color = (0, 255, 255)  # Yellow
                elif color_name == "white":
                    color = (255, 255, 255)  # White
                elif color_name == "black":
                    color = (0, 0, 0)  # Black
                else:
                    color = (128, 128, 128)  # Gray for unknown
                
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                
                # Display color information
                label = f"{color_name} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        cv2.imshow("video", frame)
        if cv2.waitKey(40) & 0xFF == ord('q'):
            break

vid.release()
cv2.destroyAllWindows()


0: 384x640 7 cars, 54.3ms
Speed: 3.5ms preprocess, 54.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 49.1ms
Speed: 1.8ms preprocess, 49.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 46.3ms
Speed: 1.7ms preprocess, 46.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 1 truck, 47.6ms
Speed: 1.8ms preprocess, 47.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 1 truck, 58.7ms
Speed: 1.6ms preprocess, 58.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 truck, 50.8ms
Speed: 1.6ms preprocess, 50.8ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 cars, 1 truck, 51.5ms
Speed: 2.9ms preprocess, 51.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 cars, 1 truck, 52.8ms
Speed: 1.9ms preprocess, 52.8ms inference, 1.1ms postprocess

In [11]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

# Updated color ranges for car paint colors (more muted tones)
COLOR_RANGES = {
    'red': [(0, 30, 30), (10, 200, 200), (170, 30, 30), (180, 200, 200)], 
    'blue': [(90, 30, 30), (140, 200, 200)],
    'green': [(35, 30, 30), (85, 200, 200)],
    'yellow': [(15, 30, 30), (35, 200, 200)],
    'orange': [(10, 30, 30), (25, 200, 200)],
    'white': [(0, 0, 180), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 40)],
    'gray': [(0, 0, 40), (180, 30, 180)],
    'silver': [(0, 0, 100), (180, 15, 220)],
    'brown': [(5, 30, 30), (20, 150, 150)],
}

def get_car_color(hsv_roi):
    """Get the car color from HSV ROI with car-specific color detection"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Remove very small ROIs to avoid noise
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return "unknown"
    
    # Calculate median HSV values (more robust than mean for color detection)
    median_h = np.median(hsv_roi[:, :, 0])
    median_s = np.median(hsv_roi[:, :, 1])
    median_v = np.median(hsv_roi[:, :, 2])
    
    # Convert to float for comparison
    h, s, v = float(median_h), float(median_s), float(median_v)
    
    # Color detection logic for car paints
    if v < 45:
        return "black"
    elif s < 25 and v > 180:
        return "white"
    elif s < 30 and 100 < v < 220:
        return "silver"
    elif s < 35 and 40 < v < 180:
        return "gray"
    elif (0 <= h <= 10) or (170 <= h <= 180):
        if s > 40 and v > 60:
            return "red"
        else:
            return "brown"
    elif 10 < h <= 25:
        if s > 40:
            return "orange"
        else:
            return "brown"
    elif 25 < h <= 40:
        return "yellow"
    elif 40 < h <= 85:
        return "green"
    elif 85 < h <= 140:
        return "blue"
    elif 140 < h < 170:
        # Purple/magenta range - less common for cars but possible
        return "purple"
    else:
        return "unknown"

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])  # Vehicles: car, truck, motorcycle, bus

        for result in results:
            boxes = result.boxes.xyxy
            confs = result.boxes.conf
            classIds = result.boxes.cls

            for box, conf, classId in zip(boxes, confs, classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Add small padding to avoid edges
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                # Extract ROI from HSV frame (avoid edges)
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get car color
                color_name = get_car_color(roi_hsv)
                
                # Get display properties
                bgr_color, display_text = get_color_display_properties(color_name)
                
                # Draw rectangle with color-specific bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display color information
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)

        cv2.imshow("video", frame)
        if cv2.waitKey(40) & 0xFF == ord('q'):
            break

vid.release()
cv2.destroyAllWindows()


0: 384x640 2 cars, 47.9ms
Speed: 2.0ms preprocess, 47.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 58.6ms
Speed: 1.5ms preprocess, 58.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 51.7ms
Speed: 1.1ms preprocess, 51.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 49.2ms
Speed: 1.0ms preprocess, 49.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 46.7ms
Speed: 2.4ms preprocess, 46.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 46.6ms
Speed: 1.1ms preprocess, 46.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 50.9ms
Speed: 1.2ms preprocess, 50.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 52.4ms
Speed: 1.5ms preprocess, 52.4ms inference, 1.0ms postprocess

KeyboardInterrupt: 

In [12]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

# Updated color ranges for car paint colors (more muted tones)
COLOR_RANGES = {
    'red': [(0, 30, 30), (10, 200, 200), (170, 30, 30), (180, 200, 200)],
    'blue': [(90, 30, 30), (140, 200, 200)],
    'green': [(35, 30, 30), (85, 200, 200)],
    'yellow': [(15, 30, 30), (35, 200, 200)],
    'orange': [(10, 30, 30), (25, 200, 200)],
    'white': [(0, 0, 180), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 40)],
    'gray': [(0, 0, 40), (180, 30, 180)],
    'silver': [(0, 0, 100), (180, 15, 220)],
    'brown': [(5, 30, 30), (20, 150, 150)],
}

def non_max_suppression(boxes, confs, classIds, iou_threshold=0.5):
    """Apply Non-Maximum Suppression to remove duplicate detections"""
    if len(boxes) == 0:
        return [], [], []
    
    # Convert to numpy arrays
    boxes = np.array(boxes)
    confs = np.array(confs)
    classIds = np.array(classIds)
    
    # Get indices of boxes sorted by confidence (descending)
    indices = np.argsort(confs)[::-1]
    
    keep_indices = []
    
    while len(indices) > 0:
        # Take the box with highest confidence
        current_idx = indices[0]
        keep_indices.append(current_idx)
        
        if len(indices) == 1:
            break
        
        # Get the current box
        current_box = boxes[current_idx]
        
        # Remove current index from list
        indices = indices[1:]
        
        # Calculate IoU with remaining boxes
        remaining_boxes = boxes[indices]
        
        # Calculate intersection coordinates
        x1 = np.maximum(current_box[0], remaining_boxes[:, 0])
        y1 = np.maximum(current_box[1], remaining_boxes[:, 1])
        x2 = np.minimum(current_box[2], remaining_boxes[:, 2])
        y2 = np.minimum(current_box[3], remaining_boxes[:, 3])
        
        # Calculate intersection area
        intersection_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        
        # Calculate union area
        current_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
        remaining_areas = (remaining_boxes[:, 2] - remaining_boxes[:, 0]) * (remaining_boxes[:, 3] - remaining_boxes[:, 1])
        union_area = current_area + remaining_areas - intersection_area
        
        # Calculate IoU
        iou = intersection_area / (union_area + 1e-6)
        
        # Keep boxes with IoU less than threshold
        indices = indices[iou < iou_threshold]
    
    return boxes[keep_indices], confs[keep_indices], classIds[keep_indices]

def get_car_color(hsv_roi):
    """Get the car color from HSV ROI with car-specific color detection"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Remove very small ROIs to avoid noise
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return "unknown"
    
    # Calculate median HSV values (more robust than mean for color detection)
    median_h = np.median(hsv_roi[:, :, 0])
    median_s = np.median(hsv_roi[:, :, 1])
    median_v = np.median(hsv_roi[:, :, 2])
    
    # Convert to float for comparison
    h, s, v = float(median_h), float(median_s), float(median_v)
    
    # Color detection logic for car paints
    if v < 45:
        return "black"
    elif s < 25 and v > 180:
        return "white"
    elif s < 30 and 100 < v < 220:
        return "silver"
    elif s < 35 and 40 < v < 180:
        return "gray"
    elif (0 <= h <= 10) or (170 <= h <= 180):
        if s > 40 and v > 60:
            return "red"
        else:
            return "brown"
    elif 10 < h <= 25:
        if s > 40:
            return "orange"
        else:
            return "brown"
    elif 25 < h <= 40:
        return "yellow"
    elif 40 < h <= 85:
        return "green"
    elif 85 < h <= 140:
        return "blue"
    elif 140 < h < 170:
        return "purple"
    else:
        return "unknown"

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])  # Vehicles: car, truck, motorcycle, bus

        all_boxes = []
        all_confs = []
        all_classIds = []
        
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classIds = result.boxes.cls.cpu().numpy()
            
            all_boxes.extend(boxes)
            all_confs.extend(confs)
            all_classIds.extend(classIds)
        
        # Apply Non-Maximum Suppression to remove duplicate detections
        if len(all_boxes) > 0:
            filtered_boxes, filtered_confs, filtered_classIds = non_max_suppression(
                all_boxes, all_confs, all_classIds, iou_threshold=0.5
            )
            
            for box, conf, classId in zip(filtered_boxes, filtered_confs, filtered_classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Add small padding to avoid edges
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                # Extract ROI from HSV frame (avoid edges)
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get car color
                color_name = get_car_color(roi_hsv)
                
                # Get display properties
                bgr_color, display_text = get_color_display_properties(color_name)
                
                # Draw rectangle with color-specific bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display color information
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)
                
                # Display class name (optional)
                class_names = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
                class_name = class_names.get(int(classId), 'vehicle')
                cv2.putText(frame, class_name, (x1, y1-30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr_color, 1)

        cv2.imshow("video", frame)
        if cv2.waitKey(40) & 0xFF == ord('q'):
            break

vid.release()
cv2.destroyAllWindows()


0: 384x640 2 cars, 55.3ms
Speed: 2.8ms preprocess, 55.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 42.8ms
Speed: 1.2ms preprocess, 42.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 81.9ms
Speed: 3.9ms preprocess, 81.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 46.8ms
Speed: 1.1ms preprocess, 46.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 50.5ms
Speed: 1.0ms preprocess, 50.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 44.1ms
Speed: 1.3ms preprocess, 44.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 57.3ms
Speed: 1.8ms preprocess, 57.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 truck, 45.5ms
Speed: 1.3ms preprocess, 45.5ms inference, 1.3ms postprocess

KeyboardInterrupt: 

In [15]:
vid.release()
cv2.destroyAllWindows()

In [19]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO('yolov8n.pt')

vid = cv2.VideoCapture(r"D:\git\learning-cv\edi\2347-157269979_tiny.mp4")

# Get video properties for output video
frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(vid.get(cv2.CAP_PROP_FPS))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video = cv2.VideoWriter('output_video_with_detections2.mp4', fourcc, fps, (frame_width, frame_height))

# Updated color ranges for car paint colors (more muted tones)
COLOR_RANGES = {
    'red': [(0, 30, 30), (10, 200, 200), (170, 30, 30), (180, 200, 200)],
    'blue': [(90, 30, 30), (140, 200, 200)],
    'green': [(35, 30, 30), (85, 200, 200)],
    'yellow': [(15, 30, 30), (35, 200, 200)],
    'orange': [(10, 30, 30), (25, 200, 200)],
    'white': [(0, 0, 180), (180, 30, 255)],
    'black': [(0, 0, 0), (180, 255, 40)],
    'gray': [(0, 0, 40), (180, 30, 180)],
    'silver': [(0, 0, 100), (180, 15, 220)],
    'brown': [(5, 30, 30), (20, 150, 150)],
}

def non_max_suppression(boxes, confs, classIds, iou_threshold=0.5):
    """Apply Non-Maximum Suppression to remove duplicate detections"""
    if len(boxes) == 0:
        return [], [], []
    
    # Convert to numpy arrays
    boxes = np.array(boxes)
    confs = np.array(confs)
    classIds = np.array(classIds)
    
    # Get indices of boxes sorted by confidence (descending)
    indices = np.argsort(confs)[::-1]
    
    keep_indices = []
    
    while len(indices) > 0:
        # Take the box with highest confidence
        current_idx = indices[0]
        keep_indices.append(current_idx)
        
        if len(indices) == 1:
            break
        
        # Get the current box
        current_box = boxes[current_idx]
        
        # Remove current index from list
        indices = indices[1:]
        
        # Calculate IoU with remaining boxes
        remaining_boxes = boxes[indices]
        
        # Calculate intersection coordinates
        x1 = np.maximum(current_box[0], remaining_boxes[:, 0])
        y1 = np.maximum(current_box[1], remaining_boxes[:, 1])
        x2 = np.minimum(current_box[2], remaining_boxes[:, 2])
        y2 = np.minimum(current_box[3], remaining_boxes[:, 3])
        
        # Calculate intersection area
        intersection_area = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
        
        # Calculate union area
        current_area = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
        remaining_areas = (remaining_boxes[:, 2] - remaining_boxes[:, 0]) * (remaining_boxes[:, 3] - remaining_boxes[:, 1])
        union_area = current_area + remaining_areas - intersection_area
        
        # Calculate IoU
        iou = intersection_area / (union_area + 1e-6)
        
        # Keep boxes with IoU less than threshold
        indices = indices[iou < iou_threshold]
    
    return boxes[keep_indices], confs[keep_indices], classIds[keep_indices]

def get_car_color(hsv_roi):
    """Get the car color from HSV ROI with car-specific color detection"""
    if hsv_roi.size == 0:
        return "unknown"
    
    # Remove very small ROIs to avoid noise
    if hsv_roi.shape[0] < 5 or hsv_roi.shape[1] < 5:
        return "unknown"
    
    # Calculate median HSV values (more robust than mean for color detection)
    median_h = np.median(hsv_roi[:, :, 0])
    median_s = np.median(hsv_roi[:, :, 1])
    median_v = np.median(hsv_roi[:, :, 2])
    
    # Convert to float for comparison
    h, s, v = float(median_h), float(median_s), float(median_v)
    
    # Color detection logic for car paints
    if v < 45:
        return "black"
    elif s < 25 and v > 180:
        return "white"
    elif s < 30 and 100 < v < 220:
        return "silver"
    elif s < 35 and 40 < v < 180:
        return "gray"
    elif (0 <= h <= 10) or (170 <= h <= 180):
        if s > 40 and v > 60:
            return "red"
        else:
            return "brown"
    elif 10 < h <= 25:
        if s > 40:
            return "orange"
        else:
            return "brown"
    elif 25 < h <= 40:
        return "yellow"
    elif 40 < h <= 85:
        return "green"
    elif 85 < h <= 140:
        return "blue"
    elif 140 < h < 170:
        return "purple"
    else:
        return "unknown"

def get_color_display_properties(color_name):
    """Return display color and text for different car colors"""
    color_map = {
        'red': ((0, 0, 255), "RED"),
        'blue': ((255, 0, 0), "BLUE"),
        'green': ((0, 255, 0), "GREEN"),
        'yellow': ((0, 255, 255), "YELLOW"),
        'orange': ((0, 165, 255), "ORANGE"),
        'white': ((255, 255, 255), "WHITE"),
        'black': ((0, 0, 0), "BLACK"),
        'gray': ((128, 128, 128), "GRAY"),
        'silver': ((192, 192, 192), "SILVER"),
        'brown': ((42, 42, 165), "BROWN"),
        'purple': ((128, 0, 128), "PURPLE"),
        'unknown': ((128, 128, 128), "UNKNOWN")
    }
    return color_map.get(color_name, ((128, 128, 128), "UNKNOWN"))

# Counter for progress display
frame_count = 0
total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

ret = True
while ret:
    ret, frame = vid.read()

    if ret:
        frame_count += 1
        print(f"Processing frame {frame_count}/{total_frames} ({frame_count/total_frames*100:.1f}%)")
        
        # Convert frame to HSV for color analysis
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        results = model(frame, classes=[1, 2, 3, 5, 7])  # Vehicles: car, truck, motorcycle, bus

        all_boxes = []
        all_confs = []
        all_classIds = []
        
        for result in results:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classIds = result.boxes.cls.cpu().numpy()
            
            all_boxes.extend(boxes)
            all_confs.extend(confs)
            all_classIds.extend(classIds)
        
        # Apply Non-Maximum Suppression to remove duplicate detections
        if len(all_boxes) > 0:
            filtered_boxes, filtered_confs, filtered_classIds = non_max_suppression(
                all_boxes, all_confs, all_classIds, iou_threshold=0.5
            )
            
            for box, conf, classId in zip(filtered_boxes, filtered_confs, filtered_classIds):
                x1, y1, x2, y2 = map(int, box)
                
                # Add small padding to avoid edges
                padding = 5
                x1_pad = max(0, x1 + padding)
                y1_pad = max(0, y1 + padding)
                x2_pad = min(frame.shape[1], x2 - padding)
                y2_pad = min(frame.shape[0], y2 - padding)
                
                # Extract ROI from HSV frame (avoid edges)
                roi_hsv = hsv_frame[y1_pad:y2_pad, x1_pad:x2_pad]
                
                # Get car color
                color_name = get_car_color(roi_hsv)
                
                # Get display properties
                bgr_color, display_text = get_color_display_properties(color_name)
                
                # Draw rectangle with color-specific bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), bgr_color, 2)
                
                # Display color information
                label = f"{display_text} ({conf:.2f})"
                cv2.putText(frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, bgr_color, 2)
                
                # Display class name
                class_names = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
                class_name = class_names.get(int(classId), 'vehicle')
                cv2.putText(frame, class_name, (x1, y1-30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr_color, 1)

        # Write the frame to output video
        output_video.write(frame)
        
        # Display progress (optional)
        cv2.imshow("Processing Video - Press 'q' to stop", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Release everything
vid.release()
output_video.release()
cv2.destroyAllWindows()

print(f"Video processing complete! Output saved as 'output_video_with_detections.avi'")

Processing frame 1/1274 (0.1%)

0: 384x640 2 cars, 53.0ms
Speed: 1.8ms preprocess, 53.0ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 2/1274 (0.2%)

0: 384x640 2 cars, 75.6ms
Speed: 1.2ms preprocess, 75.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 3/1274 (0.2%)

0: 384x640 2 cars, 56.0ms
Speed: 1.9ms preprocess, 56.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 4/1274 (0.3%)

0: 384x640 2 cars, 1 truck, 51.2ms
Speed: 1.3ms preprocess, 51.2ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 5/1274 (0.4%)

0: 384x640 2 cars, 1 truck, 58.2ms
Speed: 1.2ms preprocess, 58.2ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 6/1274 (0.5%)

0: 384x640 2 cars, 1 truck, 50.7ms
Speed: 1.2ms preprocess, 50.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
Processing frame 7/1274 (0.5%)

0: 384x640 